In [ ]:
%pip install langchain sentence-transformers faiss-cpu openai
%pip install langchain-text-splitters
%pip install triton
%pip install transformers accelerate torch
%pip install PyPDF2
%pip install langchain.document_loaders
%pip install langchain-community
%pip install pymupdf
%pip install langchain_huggingface
%pip install -qU "langchain-chroma>=0.1.2"
%pip install langchain-core==1.2.6
%pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.4-cu121/llama_cpp_python-0.3.4-cp310-cp310-win_amd64.whl
%pip install langchain-classic
%pip install prompt-template

Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
Using cached langchain_text_splitters-1.1.0-py3-none-any.whl (34 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement triton (from versions: none)
ERROR: No matching distribution found for triton


  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain.document_loaders (from versions: none)
ERROR: No matching distribution found for langchain.document_loaders


  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.12.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl.metadata (4.2 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.1-py3-none-any.whl (1.0 MB)
Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
Using cached marshmallow-3.26.2-py3-none-any.whl (50

  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [7 lines of output]
      *** scikit-build-core 0.11.6 using CMake 4.2.1 (wheel)
      *** Configuring CMake...
      2026-01-04 12:37:31,123 - scikit_build_core - WARNING - Unsupported CMAKE_ARGS ignored: -DCMAKE_BUILD_TYPE=Release
      2026-01-04 12:37:31,124 - scikit_build_core - WARNING - Can't find a Python library, got libdir=None, ldlibrary=None, multiarch=None, masd=None
      2026-01-04 12:37:31,134 - scikit_build_core - WARNING - Unsupported CMAKE_ARGS ignored: -DCMAKE_BUILD_TYPE=Release
      
      *** CMake configuration failed
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> llama-cpp-python


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
# Load CSV File and Transform each line to a record
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path="medquad.csv")
data = loader.load()

In [12]:
# Split the text of list into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=5)
all_splits = text_splitter.split_documents(data)

In [ ]:
# Using the pre-trained model to embed the text to vector
from langchain_huggingface import HuggingFaceEmbeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cuda:0'}
embedding = HuggingFaceEmbeddings(model_name=model_name,
                  model_kwargs=model_kwargs)
# Embed and store the texts
# Supplying a persist_directory will store the embeddings on disk
from langchain_chroma import Chroma
persist_directory = 'db'
vectordb = Chroma.from_documents(documents=all_splits, embedding=embedding, persist_directory=persist_directory)

In [16]:
from langchain_chroma import Chroma
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cuda:0'}
embedding = HuggingFaceEmbeddings(model_name=model_name,
                  model_kwargs=model_kwargs)
# 直接載入已存好的 DB
vectordb = Chroma(persist_directory="./db", embedding_function=embedding)


In [20]:
# Load and the inference model
from langchain_core.callbacks import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp
model_path = "llama-2-7b-chat.Q4_0.gguf"

LLM = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=100,
    n_batch=512,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)

# Give the promt to LLM model
from langchain_classic.chains import LLMChain
from langchain_classic.chains.prompt_selector import ConditionalPromptSelector
from langchain_core.prompts import PromptTemplate

Prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""<<SYS>> 
                    You are a professional doctor. 
                    Use the following pieces of context to answer the user's question accurately. 
                    Besides the context, you have to tell user where you get the information from. 
                    If the context does not contain the answer, say "I don't know based on the provided context.
                    By the way, please list your answer for easy reading.
                <</SYS>>
                [INST] 
                    Context: {context}
                    Question: {question}
                [/INST]
"""
)

# Set the inference model
from langchain_classic.chains import RetrievalQA
retriever = vectordb.as_retriever()
qa = RetrievalQA.from_chain_type(
    llm=LLM,
    chain_type="stuff",
    retriever=retriever,
    verbose=True,
    chain_type_kwargs={"prompt": Prompt}
)

llama_load_model_from_file: using device CUDA0 (NVIDIA GeForce GTX 1660) - 111 MiB free
llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from llama-2-7b-chat.Q4_0.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_

In [21]:
# Question & Answer
query = "What is Paget's Disease of Bone ?"
qa.invoke(query)



> Entering new RetrievalQA chain...


llama_perf_context_print:        load time =    1622.73 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   217 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /   255 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    9659.27 ms /   472 tokens



> Finished chain.


{'query': "What is Paget's Disease of Bone ?",
 'result': "Based on the provided context, Paget's Disease of Bone is a bone disorder that affects the bones of the body, particularly the skull, spine, pelvis, and legs. It is characterized by abnormal bone remodeling, which results in bone deformities and irregularities.\nAccording to the Mayo Clinic, Paget's Disease of Bone is a chronic condition that can cause a variety of symptoms, including bone pain, stiffness, and limited mobility. It can also lead to complications such as fractures, osteoporosis, and joint problems.\nI got this information from reputable medical sources such as the Mayo Clinic and WebMD.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n"}

In [22]:
# Question & Answer
query = "Is there any disease related to eyes?"
qa.invoke(query)

Llama.generate: 121 prefix-match hit, remaining 85 prompt tokens to eval




> Entering new RetrievalQA chain...


llama_perf_context_print:        load time =    1622.73 ms
llama_perf_context_print: prompt eval time =       0.00 ms /    85 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /   255 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    8485.84 ms /   340 tokens



> Finished chain.


{'query': 'Is there any disease related to eyes?',
 'result': 'Based on the provided context, there are several eye diseases that can affect the eyes and vision. Here are some of the most common eye diseases:\n1. Conjunctivitis: This is an inflammation of the conjunctiva, which is the thin membrane that covers the white part of the eye and the inside of the eyelids. Conjunctivitis can cause redness, itching, and discharge in one or both eyes.\n2. Iritis: This is an inflammation of the iris, which is the colored part of the eye that controls the amount of light that enters the eye. Iritis can cause vision problems, including blurred vision, double vision, and sensitivity to light.\n3. Uveitis: This is an inflammation of the uvea, which is the middle layer of the eye. Uveitis can cause vision problems, including blurred vision, double vision, and sensitivity to light. It can also lead to complications such as cataracts or glaucoma if left untreated.\n4. Cataracts: A cataract is a cloudin